# Stage 4 — Persistence race model and proposed process-anchored model (+ ablations)
M5: diffusion–censoring race (Ranger et al., 2026, version 1: censored responses are random guesses).
M6a: M5 + fast-guess disengagement mixture. M6b: M6a + non-decision time anchored on time-to-first-action (F). M6: M6b + engagement predicted by process indicators (no action, number of actions) + latent regression on AI use with in-model imputation and MNAR missingness model.
Same data, estimator and LOO scale as Stage 3. T4 GPU → Run all. Resumable.

In [ ]:
!pip -q install numpyro multipledispatch --no-deps
import os, time, json, numpy as np, pandas as pd, jax, jax.numpy as jnp, numpyro, numpyro.distributions as dist, optax, arviz as az
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
from google.colab import drive
drive.mount('/content/drive'); assert jax.devices()[0].platform=='gpu', 'Select T4 GPU'
S2='/content/drive/MyDrive/PISA2025_stage2'; S3='/content/drive/MyDrive/PISA2025_stage3'; OUT='/content/drive/MyDrive/PISA2025_stage4'; os.makedirs(OUT,exist_ok=True)
CNTS=['ARM','KAZ','MDA','LVA','LTU','EST','JPN','ARE','SGP']; SEED=20251025; STEPS=20000; NDRAW=200; EPS=0.01

D=pd.read_parquet(f'{S2}/sample_math.parquet')
ii,items=pd.factorize(D.item); pp,pers=pd.factorize(D.CNTSTUID)
pc=D.drop_duplicates('CNTSTUID').set_index('CNTSTUID').loc[pers]; cp=pd.Categorical(pc.CNT,categories=CNTS).codes
ts=D.t.values/10.; g=pd.Series(ts).groupby(ii)
fs=D.f.values/10.; hasf=~np.isnan(fs); A=D.A.values
esc=pc.ESCS.fillna(pc.groupby('CNT').ESCS.transform('mean')).fillna(0).values; fem=(pc.ST004D01T.values==1).astype(np.float32)
ai=pc.AIUSESCH.values; aiobs=~np.isnan(ai)
d=dict(ii=jnp.array(ii),pp=jnp.array(pp),cp=jnp.array(cp),x=jnp.array(D.x.values,jnp.float32),ts=jnp.array(ts,jnp.float32),
       mint=jnp.array(g.min().values,jnp.float32),tmax=jnp.array(g.max().values,jnp.float32),
       fs=jnp.array(np.nan_to_num(fs),jnp.float32),hasf=jnp.array(hasf),noact=jnp.array(~hasf,jnp.float32),lA=jnp.array(np.log1p(np.nan_to_num(A)),jnp.float32),
       ai=jnp.array(np.nan_to_num(ai),jnp.float32),aiobs=jnp.array(aiobs),esc=jnp.array(esc,jnp.float32),fem=jnp.array(fem))
nI,nP,nC=len(items),len(pers),len(CNTS); print(nI,nP,len(D),'AI observed',aiobs.mean().round(3))

def wfpt(t,v,a,w=0.5):
    u=t/a**2; k=jnp.arange(1,21.)
    sl=jnp.sum(k*jnp.sin(k*jnp.pi*w)*jnp.exp(-(k**2-1)*jnp.pi**2*u[...,None]/2),-1)
    large=jnp.log(jnp.pi)+jnp.log(jnp.maximum(sl,1e-20))-jnp.pi**2*u/2
    ww=w+2*jnp.arange(-5,6.); ss=jnp.sum(ww*jnp.exp(-(ww**2-w**2)/(2*u[...,None])),-1)
    small=-0.5*jnp.log(2*jnp.pi*u**3)+jnp.log(jnp.maximum(ss,1e-20))-w**2/(2*u)
    return jnp.where(u<0.5,small,large)-2*jnp.log(a)-v*a*w-v**2*t/2
def wsurv(t,v,a):                                                       # log P(no boundary hit by t), w=0.5 (large-time series)
    k=jnp.arange(1,151.); sk=k*jnp.sin(k*jnp.pi/2); lam=v[...,None]**2/2+k**2*jnp.pi**2/(2*a[...,None]**2)
    s=jnp.sum(sk/lam*jnp.exp(-(lam-lam[...,:1])*t[...,None]),-1)
    ls=jnp.log(2*jnp.pi/a**2)+(jnp.abs(v*a/2)+jnp.log1p(jnp.exp(-jnp.abs(v*a)))-jnp.log(2.))+jnp.log(jnp.maximum(s,1e-30))-lam[...,0]*t
    return jnp.where(t/a**2<0.01,0.,jnp.minimum(ls,0.))

DIM={'M5_race':3,'M6a_mix':4,'M6b_anchor':4,'M6_full':4}           # person dims: drift, log caution, persistence, disengagement
def eta_of(p,d,name):
    mu=jnp.concatenate([jnp.zeros((1,DIM[name])),p['mu_free']],0)[d['cp']]
    if name=='M6_full':
        aif=jnp.where(d['aiobs'],d['ai'],p['mai'][d['cp']]+p['sai']*p['ai_z'])
        mu=mu+jnp.stack([aif,d['esc'],d['fem']],1)@p['Gam'].T
    return mu+p['z']@(p['sd'][:,None]*p['Lcorr']).T

def LL(p,d,name):
    e=eta_of(p,d,name); i,q=d['ii'],d['pp']; t=d['ts']
    v=jnp.clip(e[q,0]-p['b'][i],-8.,8.); a=jnp.exp(jnp.clip(p['la'][i]+e[q,1],-3.,4.))
    ter_item=d['mint'][i]*jax.nn.sigmoid(p['r'][i])*0.99
    ter=jnp.where(d['hasf'],jax.nn.sigmoid(p['rphi'][i])*d['fs'],ter_item) if name in('M6b_anchor','M6_full') else ter_item
    td=jnp.maximum(t-ter,1e-4)
    lw=jnp.where(d['x']>0.5,wfpt(td,-v,a),wfpt(td,v,a))
    lt=jnp.log(t); zc=(lt-e[q,2]-p['oc'][i])/p['sc']                     # persistence (censoring) time ~ LogNormal
    lfc=dist.Normal(0.,1.).log_prob(zc)-jnp.log(p['sc'])-lt; lSc=jax.scipy.special.log_ndtr(-zc)
    lg=dist.Bernoulli(probs=p['g']).log_prob(d['x'])
    eng=jnp.logaddexp(lw+lSc, lfc+wsurv(td,v,a)+lg)
    if name!='M5_race':
        k=p['kap0']+e[q,3]+(p['lam'][0]*d['noact']+p['lam'][1]*d['lA'] if name=='M6_full' else 0.)
        rg=lg+dist.LogNormal(p['mur'],p['sr']).log_prob(t)
        eng=jnp.logaddexp(jax.nn.log_sigmoid(-k)+eng,jax.nn.log_sigmoid(k)+rg)
    lc=jnp.log(0.5)-jnp.log(d['tmax'][i])
    return jnp.logaddexp(jnp.log(1-EPS)+eng,jnp.log(EPS)+lc)-jnp.log(10.)

def make_model(name):
    K=DIM[name]
    def model(d):
        S=numpyro.sample; p={}
        p['mu_free']=S('mu_free',dist.Normal(0.,1.).expand([nC-1,K])); p['Lcorr']=S('Lcorr',dist.LKJCholesky(K,2.))
        p['sd']=S('sd',dist.HalfNormal(1.).expand([K])); p['z']=S('z',dist.Normal(0.,1.).expand([nP,K]))
        p['b']=S('b',dist.Normal(0.,2.).expand([nI])); p['la']=S('la',dist.Normal(1.,1.).expand([nI])); p['r']=S('r',dist.Normal(0.,1.5).expand([nI]))
        p['oc']=S('oc',dist.Normal(1.5,1.).expand([nI])); p['sc']=S('sc',dist.HalfNormal(1.)); p['g']=S('g',dist.Beta(2.,6.))
        if name!='M5_race': p['kap0']=S('kap0',dist.Normal(-3.,1.)); p['mur']=S('mur',dist.Normal(-1.3,.5)); p['sr']=S('sr',dist.HalfNormal(.5))
        if name in('M6b_anchor','M6_full'): p['rphi']=S('rphi',dist.Normal(0.,1.5).expand([nI]))
        if name=='M6_full':
            p['lam']=S('lam',dist.Normal(0.,2.).expand([2])); p['Gam']=S('Gam',dist.Normal(0.,1.).expand([K,3]))
            p['mai']=S('mai',dist.Normal(0.,1.).expand([nC])); p['sai']=S('sai',dist.HalfNormal(1.)); p['ai_z']=S('ai_z',dist.Normal(0.,1.).expand([nP]))
            p['zeta']=S('zeta',dist.Normal(0.,2.).expand([nC])); p['zw']=S('zw',dist.Normal(0.,1.).expand([2]))
            numpyro.factor('ai_obs',jnp.where(d['aiobs'],dist.Normal(p['mai'][d['cp']],p['sai']).log_prob(d['ai']),0.).sum())
            e=eta_of(p,d,name); lm=p['zeta'][d['cp']]+p['zw'][0]*e[:,2]+p['zw'][1]*e[:,0]        # MNAR: missing AI ~ persistence, drift
            numpyro.factor('ai_miss',dist.Bernoulli(logits=lm).log_prob((~d['aiobs']).astype(jnp.float32)).sum())
        numpyro.factor('obs',LL(p,d,name).sum())
    return model

PNAMES=['drift','log_caution','persistence','disengagement']
def run(name):
    if os.path.exists(f'{OUT}/{name}_res.json'): print(name,'already done:',open(f'{OUT}/{name}_res.json').read()); return
    model=make_model(name); guide=AutoNormal(model,init_scale=0.05)
    opt=numpyro.optim.optax_to_numpyro(optax.chain(optax.clip_by_global_norm(10.),optax.adam(optax.exponential_decay(0.005,5000,0.5,end_value=0.0003))))
    t0=time.time(); res=SVI(model,guide,opt,Trace_ELBO()).run(jax.random.PRNGKey(SEED),STEPS,d,progress_bar=True,stable_update=True); rt=time.time()-t0
    Ls=np.asarray(res.losses); np.save(f'{OUT}/{name}_elbo.npy',Ls); print('finite ELBO share',np.isfinite(Ls).mean()); assert np.isfinite(Ls[-1000:]).mean()>0.9, 'non-finite ELBO'
    s=Predictive(guide,params=res.params,num_samples=NDRAW)(jax.random.PRNGKey(1),d); s={k:v for k,v in s.items() if k not in('obs','ai_obs','ai_miss')}
    pick=lambda j: jax.tree_util.tree_map(lambda a:a[j],s)
    ll=np.asarray(jax.lax.map(lambda j: LL(pick(j),d,name),jnp.arange(NDRAW)))
    idata=az.from_dict(posterior={'dummy':np.zeros((1,NDRAW))},log_likelihood={'obs':ll[None]}); idata.to_netcdf(f'{OUT}/{name}_ll.nc'); lo=az.loo(idata)
    E=np.asarray(jax.lax.map(lambda j: eta_of(pick(j),d,name),jnp.arange(NDRAW)))
    P=pc[['CNT','AIUSESCH','ESCS','ST004D01T','EFFORT1','W_FSTUWT','ST438Q01DA','ST438Q02DA','ST438Q03DA','ST438Q04DA']].copy()
    for k in range(DIM[name]): P[PNAMES[k]]=E[...,k].mean(0); P[PNAMES[k]+'_sd']=E[...,k].std(0)
    P.reset_index().to_csv(f'{OUT}/{name}_persons.csv',index=False)
    q=lambda k: np.asarray(s[k])
    out={'runtime_min':round(rt/60,1),'neg_elbo':round(float(np.nanmean(Ls[-500:])),1),'elbo_rel_change':round(float((np.nanmean(Ls[-1500:-1000])-np.nanmean(Ls[-500:]))/abs(np.nanmean(Ls[-500:]))),6),
         'elpd_loo':round(float(lo.elpd_loo),1),'se':round(float(lo.se),1),'p_loo':round(float(lo.p_loo),1),'pareto_k_gt_0.7_pct':round(100*float((lo.pareto_k.values>0.7).mean()),2),
         'g':round(float(q('g').mean()),3),'sc':round(float(q('sc').mean()),3)}
    if name in('M6b_anchor','M6_full'): out['phi_median']=round(float(np.median(jax.nn.sigmoid(q('rphi').mean(0)))),3)
    if name!='M5_race': out['rapid_mean_sec']=round(float(10*np.exp(q('mur').mean())),2)
    if name=='M6_full':
        G=q('Gam'); out['lambda_noact_lA']=q('lam').mean(0).round(3).tolist(); out['zeta_persist_drift']=q('zw').mean(0).round(3).tolist()
        out['AI_effect']={PNAMES[k]:{'mean':round(float(G[:,k,0].mean()),4),'sd':round(float(G[:,k,0].std()),4),'q2.5':round(float(np.quantile(G[:,k,0],.025)),4),'q97.5':round(float(np.quantile(G[:,k,0],.975)),4)} for k in range(4)}
    json.dump(out,open(f'{OUT}/{name}_res.json','w'),indent=1); print(name,json.dumps(out))

for m in ['M5_race','M6a_mix','M6b_anchor','M6_full']: run(m)

allm={m:az.from_netcdf(f'{S3}/{m}_ll.nc') for m in ['M1_vdl','M3_mix','M4_diff']}
allm.update({m:az.from_netcdf(f'{OUT}/{m}_ll.nc') for m in DIM})
cmp=az.compare(allm,ic='loo'); cmp.to_csv(f'{OUT}/stage4_compare.csv'); print(cmp[['rank','elpd_loo','p_loo','elpd_diff','dse']].round(1)); print('STAGE4 DONE')
